In [1]:
import os
import glob
import numpy as np
import pandas as pd
import librosa

# --- CONFIGURATION ---
# PASTE YOUR FOLDER PATH HERE
DATASET_PATH = r"../data/raw/Audio/Baby Cry Dataset/" 

# MAPPING: 1 = Pain, 0 = No Pain
LABEL_MAP = {
    'belly pain': 1, 'cold_hot': 1, 'discomfort': 1,  # PAIN CLASSES
    'hungry': 0, 'tired': 0, 'burping': 0, 'lonely': 0, 'scared': 0 # NO PAIN CLASSES
}

def add_noise(data):
    """Adds random static noise"""
    noise_amp = 0.005 * np.random.uniform() * np.amax(data)
    return data + noise_amp * np.random.normal(size=data.shape)

def shift_pitch(data, sr):
    """Changes the pitch slightly (higher/lower)"""
    return librosa.effects.pitch_shift(y=data, sr=sr, n_steps=2.0)

def stretch_time(data):
    """Speeds up the audio slightly"""
    return librosa.effects.time_stretch(y=data, rate=1.2)

def extract_features(y, sr):
    """Extracts MFCCs (the main feature for audio AI)"""
    # MFCCs
    mfccs = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40).T, axis=0)
    # Mel Spectrogram
    mel = np.mean(librosa.feature.melspectrogram(y=y, sr=sr).T, axis=0)
    
    return np.hstack([mfccs, mel])

# --- MAIN LOOP ---
features = []
labels = []

print("Starting Data Processing with AUGMENTATION...")

for folder_name, label in LABEL_MAP.items():
    folder_path = os.path.join(DATASET_PATH, folder_name)
    audio_files = glob.glob(os.path.join(folder_path, "*.wav"))
    
    print(f"Processing '{folder_name}' (Class {label}). Found {len(audio_files)} files.")
    
    for file in audio_files:
        try:
            # Load Audio (limit to 5 seconds for consistency)
            y, sr = librosa.load(file, duration=5.0)
            
            # 1. Save the ORIGINAL file
            feat = extract_features(y, sr)
            features.append(feat)
            labels.append(label)
            
            # 2. IF CLASS IS PAIN (1) -> AUGMENT DATA
            # We create artificial copies to balance the dataset
            if label == 1:
                # Augment 1: Add Noise
                y_noise = add_noise(y)
                features.append(extract_features(y_noise, sr))
                labels.append(label)
                
                # Augment 2: Pitch Shift (Sound slightly higher)
                y_pitch = shift_pitch(y, sr)
                features.append(extract_features(y_pitch, sr))
                labels.append(label)
                
                # Augment 3: Time Stretch (Faster)
                y_stretch = stretch_time(y)
                features.append(extract_features(y_stretch, sr))
                labels.append(label)
                
        except Exception as e:
            print(f"Error file {file}: {e}")

# --- SAVE ---
cols = [f'mfcc_{i}' for i in range(40)] + [f'mel_{i}' for i in range(128)]
df = pd.DataFrame(features, columns=cols)
df['label'] = labels

print(f"\nPROCESSING COMPLETE!")
print(f"Total samples processed: {len(df)}")
print(f"Class Distribution:\n{df['label'].value_counts()}")

df.to_csv("../data/processed/AudioProcessed1.csv", index=False)
print("Saved to AudioProcessed1.csv")

Starting Data Processing with AUGMENTATION...
Processing 'belly pain' (Class 1). Found 126 files.
Processing 'cold_hot' (Class 1). Found 107 files.
Processing 'discomfort' (Class 1). Found 138 files.
Processing 'hungry' (Class 0). Found 382 files.
Processing 'tired' (Class 0). Found 136 files.
Processing 'burping' (Class 0). Found 118 files.
Processing 'lonely' (Class 0). Found 11 files.
Processing 'scared' (Class 0). Found 20 files.

PROCESSING COMPLETE!
Total samples processed: 2151
Class Distribution:
label
1    1484
0     667
Name: count, dtype: int64
Saved to AudioProcessed1.csv


In [2]:
import os
import glob
import numpy as np
import pandas as pd
import librosa
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# --- CONFIGURATION ---
# PASTE YOUR FOLDER PATH HERE
DATASET_PATH = r"../data/raw/Audio/Baby Cry Dataset/" 
OUTPUT_PATH = r"../data/processed/"
OUTPUT_FILE = "AudioProcessed_3Class.csv"

# --- CRITICAL UPDATE: 3-CLASS MAPPING ---
# matches Backend: {0: 'hunger', 1: 'pain', 2: 'normal'}
LABEL_MAP = {
    # CLASS 0: HUNGER
    'hungry': 0,

    # CLASS 1: PAIN
    'belly pain': 1, 
    'cold_hot': 1, 
    'discomfort': 1,

    # CLASS 2: NORMAL / OTHER
    'tired': 2, 
    'burping': 2, 
    'lonely': 2, 
    'scared': 2
}

# --- AUGMENTATION FUNCTIONS ---
def add_noise(data):
    """Adds random static noise"""
    noise_amp = 0.005 * np.random.uniform() * np.amax(data)
    return data + noise_amp * np.random.normal(size=data.shape)

def shift_pitch(data, sr):
    """Changes the pitch slightly (higher/lower)"""
    return librosa.effects.pitch_shift(y=data, sr=sr, n_steps=2.0)

def stretch_time(data):
    """Speeds up the audio slightly"""
    return librosa.effects.time_stretch(y=data, rate=1.2)

# --- FEATURE EXTRACTION (ENHANCED) ---
def extract_features(y, sr):
    """
    Extracts comprehensive features: 
    1. MFCC (Mean + Std) - Timbre
    2. Mel Spectrogram (Mean + Std) - Energy
    3. Chroma (Mean) - Pitch
    4. Spectral Contrast (Mean) - Roughness
    """
    features = []
    
    # 1. MFCCs (40)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    features.append(np.mean(mfccs, axis=1)) # Mean
    features.append(np.std(mfccs, axis=1))  # Std Dev (Critical for shaking voice)

    # 2. Mel Spectrogram (128)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    features.append(np.mean(mel, axis=1))   # Mean
    features.append(np.std(mel, axis=1))    # Std Dev
    
    # 3. Chroma (12)
    stft = np.abs(librosa.stft(y))
    chroma = librosa.feature.chroma_stft(S=stft, sr=sr)
    features.append(np.mean(chroma, axis=1)) # Mean

    # 4. Spectral Contrast (7)
    contrast = librosa.feature.spectral_contrast(S=stft, sr=sr)
    features.append(np.mean(contrast, axis=1)) # Mean
    
    # Combine all into one flat array
    return np.concatenate(features)

# --- MAIN LOOP ---
features_list = []
labels_list = []

# Ensure output directory exists
os.makedirs(OUTPUT_PATH, exist_ok=True)

print("🚀 Starting Data Processing with 3 CLASSES + AUGMENTATION...")

for folder_name, label in LABEL_MAP.items():
    folder_path = os.path.join(DATASET_PATH, folder_name)
    audio_files = glob.glob(os.path.join(folder_path, "*.wav"))
    
    # Skip if folder not found
    if not os.path.exists(folder_path):
        print(f"⚠️ Warning: Folder '{folder_name}' not found at {folder_path}")
        continue
        
    print(f"📂 Processing '{folder_name}' (Class {label}). Found {len(audio_files)} files.")
    
    for file in audio_files:
        try:
            # Load Audio (limit to 5 seconds)
            y, sr = librosa.load(file, duration=5.0)
            
            # --- 1. Original File ---
            feat = extract_features(y, sr)
            features_list.append(feat)
            labels_list.append(label)
            
            # --- 2. Augmentation Logic ---
            # Augment Hunger (0) and Pain (1) heavily to balance with Normal (2)
            # Normal (2) usually has many files (tired, burping, lonely, scared), so we augment it less or not at all
            if label in [0, 1]: 
                # Augment 1: Noise
                y_noise = add_noise(y)
                features_list.append(extract_features(y_noise, sr))
                labels_list.append(label)
                
                # Augment 2: Pitch
                y_pitch = shift_pitch(y, sr)
                features_list.append(extract_features(y_pitch, sr))
                labels_list.append(label)
                
                # Augment 3: Time Stretch
                y_stretch = stretch_time(y)
                features_list.append(extract_features(y_stretch, sr))
                labels_list.append(label)
                
        except Exception as e:
            print(f"❌ Error file {os.path.basename(file)}: {e}")

# --- SAVE ---
if len(features_list) > 0:
    # Generate generic column names
    cols = [f'feature_{i}' for i in range(len(features_list[0]))]
    
    df = pd.DataFrame(features_list, columns=cols)
    df['label'] = labels_list

    print(f"\n✅ PROCESSING COMPLETE!")
    print(f"Total samples: {len(df)}")
    print(f"Feature count per sample: {len(cols)}")
    print(f"Class Distribution:\n{df['label'].value_counts()}")

    save_loc = os.path.join(OUTPUT_PATH, OUTPUT_FILE)
    df.to_csv(save_loc, index=False)
    print(f"💾 Saved to: {save_loc}")
else:
    print("\n❌ No data processed. Check your DATASET_PATH.")

🚀 Starting Data Processing with 3 CLASSES + AUGMENTATION...
📂 Processing 'hungry' (Class 0). Found 382 files.
📂 Processing 'belly pain' (Class 1). Found 126 files.
📂 Processing 'cold_hot' (Class 1). Found 107 files.
📂 Processing 'discomfort' (Class 1). Found 138 files.
📂 Processing 'tired' (Class 2). Found 136 files.
📂 Processing 'burping' (Class 2). Found 118 files.
📂 Processing 'lonely' (Class 2). Found 11 files.
📂 Processing 'scared' (Class 2). Found 20 files.

✅ PROCESSING COMPLETE!
Total samples: 3297
Feature count per sample: 355
Class Distribution:
label
0    1528
1    1484
2     285
Name: count, dtype: int64
💾 Saved to: ../data/processed/AudioProcessed_3Class.csv


In [1]:
import os
import glob
import numpy as np
import pandas as pd
import librosa
import warnings

warnings.filterwarnings('ignore')

# --- CONFIGURATION ---
DATASET_PATH = r"../data/raw/Audio/Baby Cry Dataset/" 
OUTPUT_PATH = r"../data/processed/"
OUTPUT_FILE = "AudioProcessed_3Class_Balanced.csv"

# Matches Backend: {0: 'hunger', 1: 'pain', 2: 'normal'}
LABEL_MAP = {
    'hungry': 0,
    'belly pain': 1, 'cold_hot': 1, 'discomfort': 1,
    'tired': 2, 'burping': 2, 'lonely': 2, 'scared': 2
}

# --- AUGMENTATION FUNCTIONS ---
def add_noise(data, factor=0.005):
    noise = np.random.randn(len(data))
    return data + factor * noise

def shift_pitch(data, sr, steps):
    return librosa.effects.pitch_shift(y=data, sr=sr, n_steps=steps)

def stretch_time(data, rate):
    return librosa.effects.time_stretch(y=data, rate=rate)

# --- FEATURE EXTRACTION ---
def extract_features(y, sr):
    features = []
    # 1. MFCC (Mean + Std)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    features.append(np.mean(mfccs, axis=1))
    features.append(np.std(mfccs, axis=1))
    
    # 2. Mel Spectrogram (Mean + Std)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    features.append(np.mean(mel, axis=1))
    features.append(np.std(mel, axis=1))
    
    # 3. Chroma & Contrast
    stft = np.abs(librosa.stft(y))
    chroma = librosa.feature.chroma_stft(S=stft, sr=sr)
    contrast = librosa.feature.spectral_contrast(S=stft, sr=sr)
    features.append(np.mean(chroma, axis=1))
    features.append(np.mean(contrast, axis=1))
    
    return np.concatenate(features)

# --- MAIN LOOP ---
features_list = []
labels_list = []

print("🚀 Starting Data Processing with SMART BALANCING...")

for folder_name, label in LABEL_MAP.items():
    folder_path = os.path.join(DATASET_PATH, folder_name)
    audio_files = glob.glob(os.path.join(folder_path, "*.wav"))
    
    print(f"📂 Processing '{folder_name}' (Class {label}). Found {len(audio_files)} files.")
    
    for file in audio_files:
        try:
            y, sr = librosa.load(file, duration=5.0)
            
            # 1. Original
            features_list.append(extract_features(y, sr))
            labels_list.append(label)
            
            # --- SMART BALANCING LOGIC ---
            
            # IF CLASS 2 (Normal): It is tiny, so we AUGMENT IT 5 TIMES!
            if label == 2:
                # 1. Noise
                features_list.append(extract_features(add_noise(y), sr))
                labels_list.append(label)
                # 2. Pitch Lower
                features_list.append(extract_features(shift_pitch(y, sr, -2), sr))
                labels_list.append(label)
                 # 3. Pitch Higher
                features_list.append(extract_features(shift_pitch(y, sr, 2), sr))
                labels_list.append(label)
                # 4. Faster
                features_list.append(extract_features(stretch_time(y, 1.1), sr))
                labels_list.append(label)
                # 5. Slower
                features_list.append(extract_features(stretch_time(y, 0.9), sr))
                labels_list.append(label)

            # IF CLASS 0 or 1 (Hunger/Pain): Standard Augmentation (just 1-2 times)
            elif label in [0, 1]: 
                # 1. Noise
                features_list.append(extract_features(add_noise(y), sr))
                labels_list.append(label)
                # 2. Pitch
                features_list.append(extract_features(shift_pitch(y, sr, 2), sr))
                labels_list.append(label)
                
        except Exception as e:
            pass

# --- SAVE ---
cols = [f'feature_{i}' for i in range(len(features_list[0]))]
df = pd.DataFrame(features_list, columns=cols)
df['label'] = labels_list

print(f"\n✅ PROCESSING COMPLETE!")
print(f"Class Distribution:\n{df['label'].value_counts()}") # Check if numbers are closer now!

os.makedirs(OUTPUT_PATH, exist_ok=True)
save_loc = os.path.join(OUTPUT_PATH, OUTPUT_FILE)
df.to_csv(save_loc, index=False)
print(f"💾 Saved to: {save_loc}")

🚀 Starting Data Processing with SMART BALANCING...
📂 Processing 'hungry' (Class 0). Found 382 files.
📂 Processing 'belly pain' (Class 1). Found 126 files.
📂 Processing 'cold_hot' (Class 1). Found 107 files.
📂 Processing 'discomfort' (Class 1). Found 138 files.
📂 Processing 'tired' (Class 2). Found 136 files.
📂 Processing 'burping' (Class 2). Found 118 files.
📂 Processing 'lonely' (Class 2). Found 11 files.
📂 Processing 'scared' (Class 2). Found 20 files.

✅ PROCESSING COMPLETE!
Class Distribution:
label
2    1710
0    1146
1    1113
Name: count, dtype: int64
💾 Saved to: ../data/processed/AudioProcessed_3Class_Balanced.csv
